# Lab 07: Spark SQL, Views & Catalog

## Objectives
- Run SQL queries using spark.sql() and %sql magic
- Create and manage temporary views and global temporary views
- Explore the Unity Catalog (list catalogs, schemas, tables)
- Use SQL for complex queries: subqueries, CTEs, CASE expressions

## Exam Domain
- **Using Spark SQL — 20%**

## Estimates
- **Time:** ~35 minutes
- **Cost:** $1-2 (serverless)
- **Compute:** Serverless

![SQL, Catalog & Views](../assets/diagrams/lab-07-sql-catalog-views.png)

In [ ]:
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

## A. Running SQL Queries

Spark SQL lets you query DataFrames using standard SQL syntax. Use `spark.sql()` to execute SQL and get a DataFrame back.

In [ ]:
# Basic SQL query — returns a DataFrame
result = spark.sql("""
    SELECT PULocationID, COUNT(*) AS trips, ROUND(AVG(total_amount), 2) AS avg_fare
    FROM taxi_trips
    GROUP BY PULocationID
    ORDER BY trips DESC
    LIMIT 10
""")
result.show()

In [ ]:
# You can chain DataFrame operations on SQL results
result.filter(result.avg_fare > 30).show()

## B. Temporary Views

Temp views allow you to reference a DataFrame by name in SQL queries. They are session-scoped — they disappear when the SparkSession ends.

In [ ]:
# Create a temp view from a DataFrame
taxi_df = spark.table("taxi_trips")
taxi_df.createOrReplaceTempView("taxi")

# Query the temp view
spark.sql("SELECT COUNT(*) AS total FROM taxi").show()

In [ ]:
# createOrReplaceTempView — overwrites if exists
# createTempView — throws error if exists
github_df = spark.table("github_events")
github_df.createOrReplaceTempView("github")

spark.sql("SELECT type, COUNT(*) AS cnt FROM github GROUP BY type ORDER BY cnt DESC").show()

## C. Global Temporary Views

Global temp views are application-scoped — they survive across SparkSessions within the same Spark application. They live in the `global_temp` database.

In [ ]:
# Create a global temp view
# Note: Global temp views are NOT supported on serverless compute
# On a cluster, you would run: taxi_df.createOrReplaceGlobalTempView("taxi_global")
# And access it via: SELECT * FROM global_temp.taxi_global

try:
    taxi_df.createOrReplaceGlobalTempView("taxi_global")
    spark.sql("SELECT COUNT(*) AS total FROM global_temp.taxi_global").show()
except Exception as e:
    print(f"Global temp view not supported on this compute: {e}")
    print("On a cluster, global temp views are application-scoped and require the global_temp. prefix.")

> **Exam Tip:** Global temp views MUST be accessed with the `global_temp.` prefix. Forgetting the prefix is a common exam trap. `SELECT * FROM my_global_view` will fail — use `SELECT * FROM global_temp.my_global_view`.

## D. Exploring the Catalog

The Catalog API lets you list and inspect databases, tables, and views programmatically.

In [ ]:
# List all tables in the current schema
tables = spark.catalog.listTables()
for t in tables:
    print(f"  {t.name} (type={t.tableType}, isTemporary={t.isTemporary})")

In [ ]:
# Check if a table exists
print(f"taxi_trips exists: {spark.catalog.tableExists('taxi_trips')}")
print(f"nonexistent exists: {spark.catalog.tableExists('nonexistent')}")

# List columns of a table
columns = spark.catalog.listColumns("taxi_trips")
for c in columns:
    print(f"  {c.name}: {c.dataType} (nullable={c.nullable})")

In [ ]:
# SQL equivalents
spark.sql("SHOW TABLES").show()
spark.sql("DESCRIBE TABLE taxi_trips").show(truncate=False)
spark.sql("DESCRIBE EXTENDED taxi_trips").show(truncate=False)

## E. Advanced SQL — Subqueries and CTEs

Common Table Expressions (CTEs) and subqueries let you write complex queries in readable, composable blocks.

In [ ]:
# CTE (WITH clause) — the modern way to write complex queries
spark.sql("""
    WITH hourly_stats AS (
        SELECT
            HOUR(tpep_pickup_datetime) AS pickup_hour,
            COUNT(*) AS trips,
            ROUND(AVG(total_amount), 2) AS avg_fare
        FROM taxi_trips
        GROUP BY HOUR(tpep_pickup_datetime)
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (ORDER BY trips DESC) AS rank
        FROM hourly_stats
    )
    SELECT * FROM ranked WHERE rank <= 5
""").show()

In [ ]:
# Subquery — used in WHERE clause
spark.sql("""
    SELECT PULocationID, COUNT(*) AS trips
    FROM taxi_trips
    WHERE total_amount > (
        SELECT AVG(total_amount) FROM taxi_trips
    )
    GROUP BY PULocationID
    ORDER BY trips DESC
    LIMIT 5
""").show()

## F. CASE Expressions and Built-in Functions

In [ ]:
# CASE expression
spark.sql("""
    SELECT
        payment_type,
        CASE payment_type
            WHEN 1 THEN 'Credit Card'
            WHEN 2 THEN 'Cash'
            WHEN 3 THEN 'No Charge'
            WHEN 4 THEN 'Dispute'
            ELSE 'Unknown'
        END AS payment_label,
        COUNT(*) AS trips
    FROM taxi_trips
    GROUP BY payment_type
    ORDER BY trips DESC
""").show()

In [ ]:
# Built-in SQL functions
spark.sql("""
    SELECT
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(PERCENTILE(total_amount, 0.5), 2) AS median_fare,
        DATE_TRUNC('month', MIN(tpep_pickup_datetime)) AS earliest_month,
        DATE_TRUNC('month', MAX(tpep_pickup_datetime)) AS latest_month
    FROM taxi_trips
""").show(truncate=False)

## G. Creating Persistent Views

Unlike temp views, persistent views are stored in the catalog and survive across sessions.

In [ ]:
# Create a persistent view in the catalog
spark.sql("""
    CREATE OR REPLACE VIEW taxi_summary AS
    SELECT
        PULocationID,
        COUNT(*) AS trip_count,
        ROUND(AVG(total_amount), 2) AS avg_fare,
        ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM taxi_trips
    GROUP BY PULocationID
""")

spark.sql("SELECT * FROM taxi_summary ORDER BY trip_count DESC LIMIT 5").show()

## Exam-Style Review

**Q1.** What is the scope of a temporary view created with `createOrReplaceTempView()`?
- A) It persists in the catalog across sessions
- B) It is available to all SparkSessions in the application
- C) It is scoped to the current SparkSession only
- D) It expires after 30 minutes

**Q2.** How do you access a global temporary view named `my_view`?
- A) `SELECT * FROM my_view`
- B) `SELECT * FROM temp.my_view`
- C) `SELECT * FROM global_temp.my_view`
- D) `SELECT * FROM global.my_view`

**Q3.** What does `spark.catalog.listTables()` return?
- A) Only permanent tables
- B) Only temporary views
- C) Both permanent tables and temporary views
- D) Only Delta tables

**Q4.** What is a CTE in SQL?
- A) A type of table index
- B) A named temporary result set defined with `WITH` that exists for the duration of a single query
- C) A permanent view that auto-updates
- D) A cached table in memory

### Answers
- **Q1: C** — Temp views are scoped to the current SparkSession. They disappear when the session ends.
- **Q2: C** — Global temp views live in the `global_temp` database. You must use the `global_temp.` prefix.
- **Q3: C** — `listTables()` returns both permanent tables and temporary views in the current schema.
- **Q4: B** — A CTE (Common Table Expression) is a named temporary result set defined with `WITH`, scoped to a single query.

## Key Takeaways
- `spark.sql()` returns a DataFrame — you can chain DataFrame operations on SQL results
- Temp views are session-scoped; global temp views are application-scoped (require `global_temp.` prefix)
- `spark.catalog` provides programmatic access to tables, views, and columns
- CTEs (`WITH`) make complex SQL queries readable and composable
- Persistent views are stored in the catalog and survive across sessions

**Next:** [Lab 08 — Structured Streaming](08-structured-streaming.ipynb)

In [ ]:
spark.catalog.dropTempView("taxi")
spark.catalog.dropTempView("github")
try:
    spark.catalog.dropGlobalTempView("taxi_global")
except Exception:
    pass
# Keep the persistent view taxi_summary for later labs
print("Lab 07 cleanup complete.")